### Structured Output

"Labrador is a breed of Dog"

{ 'content':'Dog', 'species':'Labra' }

In [2]:
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import AzureOpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage

from pydantic import BaseModel
from dotenv import load_dotenv
import os

In [3]:
from azure.keyvault.secrets import SecretClient
from azure.identity import DefaultAzureCredential


load_dotenv()

vault_url = os.getenv("AZURE_VAULT_URL")

credential = DefaultAzureCredential()

client = SecretClient(credential=credential, vault_url=vault_url)

open_api_key = client.get_secret("openai-api-key").value
open_api_version = client.get_secret("openai-api-version").value
openai_deployment = client.get_secret("openai-deployment").value
openai_endpoint = client.get_secret("openai-endpoint").value

In [ ]:
model_client = AzureOpenAIChatCompletionClient(
    model="gpt-4o",
    azure_deployment=openai_deployment,
    api_key=open_api_key,
    api_version=open_api_version,
    azure_endpoint=openai_endpoint,
)

In [5]:
class PlanetInfo(BaseModel):
    name:str
    color:str
    distance_format:int

In [6]:
agent = AssistantAgent(
    name='planet_agent',
    model_client=model_client,
    system_message="You are a helpful assistant that provides information about planets. in the structure JSON" \
    "{ name :str" \
    "age : int" \
    "}"
)

Unstructured Output

In [7]:
async def test_structured_output():
    task = TextMessage(content = "Please provide information about Mars.",source='User')
    result = await agent.run(task=task)
    structured_response = result.messages[-1].content
    print(structured_response)

await test_structured_output()

```json
{
  "name": "Mars",
  "age": 4500000000
}
```


d:\All Python\Microsoft Autogen\.venv\Lib\site-packages\autogen_agentchat\agents\_assistant_agent.py:1109: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(


Strcutured Output

In [8]:
structured_model_client = AzureOpenAIChatCompletionClient(
    model="gpt-4o",
    azure_deployment=openai_deployment,
    api_key=open_api_key,
    api_version=open_api_version,
    azure_endpoint=openai_endpoint,
    response_format = PlanetInfo
)

In [10]:
agent = AssistantAgent(
    name='planet_agent',
    model_client=structured_model_client,
    system_message="You are a helpful assistant that provides information about planets. in the structure JSON"
)

In [11]:
async def test_structured_output():
    task = TextMessage(content = "Please provide information about Mars.",source='User')
    result = await agent.run(task=task)
    structured_response = result.messages[-1].content
    print(structured_response)

resultJson = await test_structured_output()

{"name":"Mars","color":"Red","distance_format":1}


d:\All Python\Microsoft Autogen\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=PlanetInfo(name='Mars', c...Red', distance_format=1), input_type=PlanetInfo])
  return self.__pydantic_serializer__.to_python(
d:\All Python\Microsoft Autogen\.venv\Lib\site-packages\autogen_agentchat\agents\_assistant_agent.py:1109: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(
